# Local incidence angle
Computes the NISAR local incidence angle (LIA) for the study box from the LOS vectors of one GSLC granule and a 10 m DEM. Change `TRACK`/`FRAME` to run another track.

In [ ]:
import logging
import sys

import matplotlib.pyplot as plt
import rioxarray

sys.path.append('..')
from scripts.lia import compute_lia

logging.basicConfig(level=logging.INFO)

# Box bounds taken from data/dem/dem_10m_testbox.tif
X, Y, CRS = (315360, 325600), (4306900, 4321100), 'EPSG:32613'
TRACK, FRAME = 19, 21

Needs Earthdata credentials. The DEM and the LIA GeoTIFF are skipped if they already exist, and the LIA goes to `data/lia`.

In [ ]:
lia_path = compute_lia(
    X, Y, CRS, TRACK, FRAME,
    start_date='2026-01-01', end_date='2026-11-01',
    resolution=10, out_dir='../data/lia', dem_path='../data/dem/dem_10m_track019_frame021.tif',
)
lia_path

Check: flat terrain should sit near the nominal NISAR incidence angle (about 30-45 degrees), and slopes facing the radar should be lower.

In [ ]:
lia = rioxarray.open_rasterio(lia_path, masked=True).squeeze(drop=True)
dem = rioxarray.open_rasterio('../data/dem/dem_10m_track019_frame021.tif', masked=True).squeeze(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharex=True, sharey=True)
dem.plot(ax=axes[0], cmap='terrain', cbar_kwargs={'label': 'Elevation (m)'})
lia.plot(ax=axes[1], cmap='viridis', cbar_kwargs={'label': 'Local incidence angle (deg)'})
axes[0].set_title('DEM'); axes[1].set_title('Local incidence angle')
for ax in axes:
    ax.set_aspect('equal')
print(float(lia.min()), float(lia.median()), float(lia.max()))